In [28]:
from pyspark.sql.functions import *

# Reading the data from S3 Bucket

In [4]:
# Define the S3 path to your data 
path_defects = "s3://manufacturing-quality-analytics/bronze/defects.csv" 
path_inspection = "s3://manufacturing-quality-analytics/bronze/inspection_logs.csv"
path_stations = "s3://manufacturing-quality-analytics/bronze/stations.csv" 
path_vehicles = "s3://manufacturing-quality-analytics/bronze/vehicles.csv" 

In [9]:
defects = spark.read.option("header", True) \
    .option("inferSchema", True) \
    .csv(path_defects)

defects.show(5)

+---------+--------------------+--------------+
|defect_id|  defect_description|severity_level|
+---------+--------------------+--------------+
|     NONE|     No defect found|          NONE|
|     D001|    Scratch on paint|         MINOR|
|     D002|    Uneven panel gap|         MINOR|
|     D003|Missing torque on...|      CRITICAL|
|     D004|    Electrical short|      CRITICAL|
+---------+--------------------+--------------+
only showing top 5 rows


In [19]:
print(f"number of rows : {defects.count()}")

number of rows : 9


In [10]:
inspection = spark.read.option("header", True) \
    .option("inferSchema", True) \
    .csv(path_inspection)
inspection.show(5)

+-------------+-----------------+----------+---------+-------------------+---------------------------+
|inspection_id|              vin|station_id|defect_id|          timestamp|inspection_duration_seconds|
+-------------+-----------------+----------+---------+-------------------+---------------------------+
|INSP-00000001|82HFE97646DEZ9600|    ST-001|     NONE|2025-01-02T14:25:49|                         13|
|INSP-00000002|82HFE97646DEZ9600|    ST-002|     NONE|2025-01-02T17:21:36|                       -325|
|INSP-00000003|82HFE97646DEZ9600|    ST-003|     NONE|2025-01-02T04:48:44|                        364|
|INSP-00000004|82HFE97646DEZ9600|    ST-004|     NONE|2025-01-02T04:59:33|                         29|
|INSP-00000005|82HFE97646DEZ9600|    ST-005|     NONE|2025-01-02T00:18:30|                         46|
+-------------+-----------------+----------+---------+-------------------+---------------------------+
only showing top 5 rows


In [20]:
print(f"number of rows : {inspection.count()}")

number of rows : 27000


In [11]:
stations = spark.read.option("header", True) \
    .option("inferSchema", True) \
    .csv(path_stations)
stations.show(5)

+----------+---------+---------------+
|station_id|zone_name|inspection_type|
+----------+---------+---------------+
|    ST-001|  Chassis|      automated|
|    ST-002|  Chassis|      automated|
|    ST-003|  Chassis|      automated|
|    ST-004|  Chassis|      automated|
|    ST-005|  Chassis|      automated|
+----------+---------+---------------+
only showing top 5 rows


In [21]:
print(f"number of rows : {stations.count()}")

number of rows : 27


In [12]:
vehicles = spark.read.option("header", True) \
    .option("inferSchema", True) \
    .csv(path_vehicles)
vehicles.show(5)

+-----------------+-------+-------------------+--------------------+
|              vin|  model|               trim|               color|
+-----------------+-------+-------------------+--------------------+
|82HFE97646DEZ9600|Model S|           Standard|Midnight Silver M...|
|2CTEVH1A3DM766542|Model 3|Standard Range Plus|Pearl White Multi...|
|R7NNG3W85PC536184|Model S|              Plaid|      Red Multi-Coat|
|EJ6E7RHW9DH5A5255|Model S|              Plaid|  Deep Blue Metallic|
|FAWRHEL40H5DL3056|Model S|           Standard|         Solid Black|
+-----------------+-------+-------------------+--------------------+
only showing top 5 rows


In [22]:
print(f"number of rows : {vehicles.count()}")

number of rows : 1000


# displying the schema of each dataframe

In [13]:
defects.printSchema()

root
 |-- defect_id: string (nullable = true)
 |-- defect_description: string (nullable = true)
 |-- severity_level: string (nullable = true)


In [14]:
inspection.printSchema()

root
 |-- inspection_id: string (nullable = true)
 |-- vin: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- defect_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- inspection_duration_seconds: integer (nullable = true)


In [15]:
stations.printSchema()

root
 |-- station_id: string (nullable = true)
 |-- zone_name: string (nullable = true)
 |-- inspection_type: string (nullable = true)


In [16]:
vehicles.printSchema()

root
 |-- vin: string (nullable = true)
 |-- model: string (nullable = true)
 |-- trim: string (nullable = true)
 |-- color: string (nullable = true)


# discovering the data

In [25]:
inspection.select(col('defect_id')).distinct().show()

+---------+
|defect_id|
+---------+
|     d002|
|     d005|
|     D003|
|     D005|
|     d001|
|  ERR-999|
|     d006|
|     d004|
|     D002|
|     d008|
|     none|
|     D004|
|     d003|
|     D006|
|     D008|
|     null|
|     d007|
|     NONE|
|     D001|
|     D007|
+---------+


In [26]:
defects.select(col('defect_id')).distinct().show()

+---------+
|defect_id|
+---------+
|     D003|
|     D005|
|     D002|
|     D004|
|     D006|
|     D008|
|     NONE|
|     D001|
|     D007|
+---------+


In [39]:
# Alias the DataFrames
inspection = inspection.alias("l")
defects = defects.alias("d")

# Perform left join
result = inspection.join(
    defects,
    upper(col("l.defect_id")) == upper(col("d.defect_id")),
    how="left"
).filter(
    (col("d.defect_id").isNull()) & 
    (col("l.defect_id").isNotNull()) & 
    (upper(col("l.defect_id")) != "NONE")
).select(
    col("l.defect_id")
).distinct()

# Show results
result.show()

+---------+
|defect_id|
+---------+
|  ERR-999|
+---------+


### Data Profiling Observations: `defect_id`

1. **Inconsistent Casing (Data Standardization Issue)**
    - **Finding:** Defect codes are not standardized. We have mixed casing for the exact same underlying defects (e.g., `D001` and `d001`, `D005` and `d005`).
    - **Impact:** If we group or join on this column as-is, the engine will treat `D001` and `d001` as two entirely different categories, throwing off all defect counts and metrics.

2. **Fragmented "No Defect" States (Missing Data Handling)**
    - **Finding:** A clean inspection is being represented in multiple ways: the uppercase string `NONE`, the lowercase string `none`, and `null` (which could be an actual SQL NULL value or the literal text string `'null'`).
    - **Impact:** Analysts trying to calculate a simple "Pass/Fail" rate will get inaccurate results unless they account for all three variations.

3. **Anomalous/Orphan Records (Referential Integrity Issue)**
    - **Finding:** There is an unexpected value: `ERR-999`.
    - **Impact:** This code does not exist in our `defects.csv` reference table. If we perform an INNER JOIN downstream, any inspection log tied to `ERR-999` will silently drop out of our reporting.

In [34]:
inspection.select(
    max(col('inspection_duration_seconds')).alias('max_duration'),
    min(col('inspection_duration_seconds')).alias('min_duration'),
    round(avg(col('inspection_duration_seconds')), 2).alias('avg_duration')
).show()

+------------+------------+------------+
|max_duration|min_duration|avg_duration|
+------------+------------+------------+
|         700|        -500|      144.33|
+------------+------------+------------+


### Data Profiling Observations: `inspection_duration_seconds`

1. **Impossible Minimums (Data Integrity Failure)**
    - **Finding:** The minimum inspection duration is exactly `-500` seconds.
    - **Impact:** Physical processes cannot take negative time. This definitively proves there is a sensor glitch, a system clock syncing issue, or upstream data corruption. Any downstream reporting using this raw column is currently invalid.

2. **Skewed Aggregations (Business Impact)**
    - **Finding:** The average inspection duration is currently sitting at `144.33` seconds.
    - **Impact:** Because we have severe negative outliers in the dataset, this average is artificially dragged down. The actual average inspection time is likely higher, meaning factory throughput is worse than this raw metric suggests.

3. **High Maximums (Potential Bottlenecks)**
    - **Finding:** The maximum duration is `700` seconds (over 11 minutes), which is massively higher than the current average.
    - **Impact:** While `700` seconds is mathematically possible, it's a huge outlier. This points directly to severe bottlenecks at specific stations that operations teams will want to investigate once the data is cleaned.

In [36]:
# Filter rows where 'timestamp' contains a slash '/'
inspection.filter(
    col("timestamp").like("%/%")
).select("timestamp").limit(10).show()

+-------------------+
|          timestamp|
+-------------------+
|01/01/2025 15:37:13|
|02/01/2025 15:11:59|
|01/01/2025 18:48:01|
|05/01/2025 15:44:11|
|02/01/2025 14:42:58|
|02/01/2025 15:11:27|
|02/01/2025 17:24:30|
|02/01/2025 19:40:51|
|01/01/2025 00:54:26|
|01/01/2025 22:51:52|
+-------------------+


### Data Profiling Observations: `timestamp`

- **Finding: Format Inconsistency (Schema Violation)**  
  The data pipeline expects standard ISO 8601 timestamps (e.g., `YYYY-MM-DDTHH:MM:SS`). However, a subset of records is arriving in a European/custom string format (`DD/MM/YYYY HH:MM:SS`) using slashes instead of dashes.

- **Impact 1: Fatal Parsing Errors or Data Loss**  
  If we attempt a simple `.cast("timestamp")` on this column as-is, the Spark engine will fail to parse these specific rows. Depending on strictness settings, this will either crash the entire data pipeline or silently convert these values to `NULL`, resulting in missing data.

- **Impact 2: Chronological Sorting Failure**  
  Because these are currently stored as raw strings (Varchar/String type), standard string sorting will fail. For example, the string `02/01/2025` (January 2nd) will incorrectly sort after `01/12/2025` (December 1st). Any downstream time-series analysis or window functions will be completely inaccurate until this is fixed.